# SPY 0DTE Opening Range Breakout — Quick Runner

Strategy in one line: SPY breaks the first-N-min opening range → buy ATM 0DTE call (long) or put (short) → exit at the first of (net profit target, net stop loss, time stop).

**This run validates the two best configs from the 30-day sweep over a full year.**

Top configs from the 30-day window:
- **Config A** — `or30 | conf=any | pt10 | sl10 | co11:30` (+61% over 22 days, 67% win rate, -17% max drawdown)
- **Config B** — `or15 | conf=none | pt3 | sl10 | co11:00` (+38% over 22 days, 68% win rate, -32% max drawdown)

Run cells top to bottom with the ▶ button. No editing needed.

## 1. Setup (just clone the repo)

In [ ]:
import os, sys
print('a) preparing...')
REPO = '/content/iron-condor'
BRANCH = 'claude/spy-options-trading-bot-ri4Ms'
if not os.path.isdir(REPO):
    print('b) cloning repo (a few seconds)...')
    !git clone --quiet -b {BRANCH} https://github.com/coolin815/iron-condor.git {REPO}
else:
    print('b) repo exists; pulling latest...')
    !cd {REPO} && git pull --quiet
print('c) adding src/ to PYTHONPATH...')
SRC = REPO + '/src'
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.chdir(REPO)
print('OK')

## 2. Paste your Polygon key

In [ ]:
import os, getpass
os.environ['POLYGON_API_KEY'] = getpass.getpass('Polygon API key: ')
print('Key loaded:', len(os.environ['POLYGON_API_KEY']), 'chars')

## 3. Year-long validation — run both top configs (one cell)

**First run will take 30–60 minutes** — Polygon needs to fetch ~250 days of SPY bars and option chains, hitting 429 rate limits along the way. Subsequent reruns finish in seconds because the cache is warm.

Saves results to two separate CSV pairs so we can compare:
- `results/year_A_*.csv` — `or30 | conf=any | pt10 | sl10 | co11:30`
- `results/year_B_*.csv` — `or15 | conf=none | pt3 | sl10 | co11:00`

In [ ]:
import shutil, os
os.makedirs('results', exist_ok=True)

print('=' * 70)
print('CONFIG A: or30 | conf=any | pt10 | sl10 | ts30 | co11:30')
print('=' * 70)
!python -m iron_condor.cli --days 365 --sweep --or-window 30 --confluence any --pt 0.10 --sl 0.10 --time-stop 30 --co 11:30
shutil.copy('results/sweep_trades.csv', 'results/year_A_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/year_A_summary.csv')

print('\n' + '=' * 70)
print('CONFIG B: or15 | conf=none | pt3 | sl10 | ts30 | co11:00')
print('=' * 70)
!python -m iron_condor.cli --days 365 --sweep --or-window 15 --confluence none --pt 0.03 --sl 0.10 --time-stop 30 --co 11:00
shutil.copy('results/sweep_trades.csv', 'results/year_B_trades.csv')
shutil.copy('results/sweep_summary.csv', 'results/year_B_summary.csv')

print('\nDone. Year results saved.')

## 4. Compare year-long performance

In [ ]:
import pandas as pd
a_sum = pd.read_csv('results/year_A_summary.csv')
b_sum = pd.read_csv('results/year_B_summary.csv')
side_by_side = pd.concat([a_sum, b_sum], ignore_index=True)
side_by_side

## 5. Trade-level stats for each config

In [ ]:
from iron_condor.metrics import analyze_timing

for label, path in [('A', 'results/year_A_trades.csv'),
                     ('B', 'results/year_B_trades.csv')]:
    print(f'\n=== Config {label} timing ===')
    trades = pd.read_csv(path)
    print(analyze_timing(trades))

In [ ]:
# Direction split per config
for label, path in [('A', 'results/year_A_trades.csv'),
                     ('B', 'results/year_B_trades.csv')]:
    trades = pd.read_csv(path)
    taken = trades[trades['exit_reason'].isin(['profit', 'stop', 'time'])]
    if not taken.empty and 'signal_direction' in taken.columns:
        print(f'\n=== Config {label}: long vs short ===')
        print(taken.groupby('signal_direction').agg(
            trades=('net_pnl', 'count'),
            win_rate=('net_pnl', lambda s: (s > 0).mean()),
            avg_pnl=('net_pnl', 'mean'),
            median_pnl=('net_pnl', 'median'),
        ).round(2))

In [ ]:
# Equity curve per config — see how each grew over the year
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
for label, path in [('A: or30/any/pt10', 'results/year_A_trades.csv'),
                     ('B: or15/none/pt3', 'results/year_B_trades.csv')]:
    trades = pd.read_csv(path)
    trades['day'] = pd.to_datetime(trades['day'])
    trades = trades.sort_values('day')
    ax.plot(trades['day'], trades['balance_after'], label=label, linewidth=1.5)
ax.axhline(1500, color='gray', linestyle='--', alpha=0.5, label='Starting $1500')
ax.set_xlabel('Date')
ax.set_ylabel('Account balance ($)')
ax.set_title('Year-long equity curves')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()